# Recommender System Comparison: SVD · NCF · LightGCN
**Dataset:** MovieLens 1M  
**Task:** Top-K implicit-feedback recommendation (BPR training)  
**Metric:** Recall@10, Precision@10, NDCG@10, HitRatio@10

| Model | Rec.@10 | Prec.@10 | NDCG@10 | HR@10 |
|-------|---------|----------|---------|-------|
| SVD | 0.1457 | 0.1158 | 0.1646 | 0.6168 |
| NCF | 0.1298 | 0.1068 | 0.1465 | 0.5810 |
| LightGCN | 0.0825 | 0.0678 | 0.0928 | 0.4322 |

## 0. Install Dependencies

In [ ]:
!pip install torch pandas scikit-learn scipy

## 1. Data Loading

In [ ]:
from google.colab import files
uploaded = files.upload()
# Upload ml-1m.zip when prompted

In [ ]:
import zipfile, os

with zipfile.ZipFile("ml-1m.zip", "r") as z:
    z.extractall(".")

print("Extracted files:", os.listdir("ml-1m"))

In [ ]:
import pandas as pd

df = pd.read_csv(
    "ml-1m/ratings.dat",
    sep="::",
    header=None,
    names=["user_id", "item_id", "rating", "timestamp"],
    engine="python"
)

print(f"Total interactions: {len(df):,}")
print(df.head())

## 2. Preprocessing

- Keep ratings ≥ 4 as **positive** implicit interactions  
- Re-encode user / item IDs to sequential indices  
- 80 / 10 / 10 train / val / test split  
- Define shared `ImplicitDataset`, negative sampler, BPR loss, and evaluation functions

In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import scipy.sparse as sp

# ── Filter to implicit positives ──────────────────────────────────────────────
df_implicit = df[df["rating"] >= 4].copy()
df_implicit["label"] = 1
print(f"Original interactions : {len(df):,}")
print(f"Positive interactions : {len(df_implicit):,}")

# ── Encode user/item IDs ──────────────────────────────────────────────────────
user2idx = {u: i for i, u in enumerate(df_implicit["user_id"].unique())}
item2idx = {it: i for i, it in enumerate(df_implicit["item_id"].unique())}
idx2user = {v: k for k, v in user2idx.items()}
idx2item = {v: k for k, v in item2idx.items()}
n_users  = len(user2idx)
n_items  = len(item2idx)
print(f"\nUsers : {n_users} | Items : {n_items}")

# ── Train / Val / Test split ──────────────────────────────────────────────────
train_df, temp_df = train_test_split(df_implicit, test_size=0.2, random_state=42)
val_df,   test_df = train_test_split(temp_df,     test_size=0.5, random_state=42)
print(f"\nTrain : {len(train_df):,} | Val : {len(val_df):,} | Test : {len(test_df):,}")

# ── Device ────────────────────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\nUsing device: {device}")

# ── Implicit Dataset ──────────────────────────────────────────────────────────
class ImplicitDataset(Dataset):
    def __init__(self, df):
        self.users = torch.tensor([user2idx[u] for u in df["user_id"]], dtype=torch.long)
        self.items = torch.tensor([item2idx[i] for i in df["item_id"]], dtype=torch.long)
    def __len__(self):        return len(self.users)
    def __getitem__(self, i): return self.users[i], self.items[i]

train_loader = DataLoader(ImplicitDataset(train_df), batch_size=1024, shuffle=True)

# ── Negative Sampling ─────────────────────────────────────────────────────────
train_user_items = train_df.groupby("user_id")["item_id"].apply(set).to_dict()
all_item_list    = np.array(list(item2idx.keys()))
n_all_items      = len(all_item_list)

def sample_negatives_vectorized(user_indices):
    neg_indices = []
    for u_idx in user_indices.numpy():
        u_raw   = idx2user[u_idx]
        pos_set = train_user_items.get(u_raw, set())
        while True:
            neg = all_item_list[np.random.randint(n_all_items)]
            if neg not in pos_set:
                neg_indices.append(item2idx[neg])
                break
    return torch.tensor(neg_indices, dtype=torch.long)

# ── BPR Loss ──────────────────────────────────────────────────────────────────
def bpr_loss(pos_scores, neg_scores):
    return -torch.log(torch.sigmoid(pos_scores - neg_scores)).mean()

# ── Ranking Evaluation for SVD / NCF ─────────────────────────────────────────
def evaluate(model, test_df, K=10, model_type="mf"):
    model.eval()
    test_user_items  = test_df.groupby("user_id")["item_id"].apply(set).to_dict()
    all_item_indices = torch.arange(n_items).to(device)
    recalls, precisions, ndcgs, hits = [], [], [], []

    with torch.no_grad():
        for user_raw, true_items in test_user_items.items():
            if user_raw not in user2idx:
                continue
            u_idx        = torch.tensor([user2idx[user_raw]], dtype=torch.long).to(device)
            true_indices = set(item2idx[i] for i in true_items if i in item2idx)
            if not true_indices:
                continue

            if model_type == "mf":
                scores = (model.user_emb(u_idx) @ model.item_emb.weight.T).squeeze()
            elif model_type == "ncf":
                u_rep  = u_idx.repeat(n_items)
                scores = model(u_rep, all_item_indices).squeeze()

            train_items       = train_user_items.get(user_raw, set())
            train_idx         = [item2idx[i] for i in train_items if i in item2idx]
            scores[train_idx] = -1e9

            topk     = torch.topk(scores, K).indices.cpu().numpy()
            topk_set = set(topk)
            hit      = len(topk_set & true_indices)

            recalls.append(hit / len(true_indices))
            precisions.append(hit / K)
            hits.append(1.0 if hit > 0 else 0.0)
            dcg  = sum(1 / np.log2(r + 2) for r, i in enumerate(topk) if i in true_indices)
            idcg = sum(1 / np.log2(i + 2) for i in range(min(len(true_indices), K)))
            ndcgs.append(dcg / idcg if idcg > 0 else 0)

    return {
        f"Recall@{K}"    : round(np.mean(recalls), 4),
        f"Precision@{K}" : round(np.mean(precisions), 4),
        f"NDCG@{K}"      : round(np.mean(ndcgs), 4),
        f"HitRatio@{K}"  : round(np.mean(hits), 4),
    }

# ── Ranking Evaluation for LightGCN ──────────────────────────────────────────
def evaluate_lgcn(model, adj, test_df, K=10):
    model.eval()
    test_user_items = test_df.groupby("user_id")["item_id"].apply(set).to_dict()
    recalls, precisions, ndcgs, hits = [], [], [], []

    with torch.no_grad():
        user_embs, item_embs = model.get_embeddings(adj)
        for user_raw, true_items in test_user_items.items():
            if user_raw not in user2idx:
                continue
            u_idx        = user2idx[user_raw]
            true_indices = set(item2idx[i] for i in true_items if i in item2idx)
            if not true_indices:
                continue
            scores            = (user_embs[u_idx] @ item_embs.T)
            train_items       = train_user_items.get(user_raw, set())
            train_idx         = [item2idx[i] for i in train_items if i in item2idx]
            scores[train_idx] = -1e9
            topk              = torch.topk(scores, K).indices.cpu().numpy()
            topk_set          = set(topk)
            hit               = len(topk_set & true_indices)
            recalls.append(hit / len(true_indices))
            precisions.append(hit / K)
            hits.append(1.0 if hit > 0 else 0.0)
            dcg  = sum(1 / np.log2(r + 2) for r, i in enumerate(topk) if i in true_indices)
            idcg = sum(1 / np.log2(i + 2) for i in range(min(len(true_indices), K)))
            ndcgs.append(dcg / idcg if idcg > 0 else 0)

    return {
        f"Recall@{K}"    : round(np.mean(recalls), 4),
        f"Precision@{K}" : round(np.mean(precisions), 4),
        f"NDCG@{K}"      : round(np.mean(ndcgs), 4),
        f"HitRatio@{K}"  : round(np.mean(hits), 4),
    }

print("Preprocessing complete. All shared utilities ready.")

## 3. Model 1 — SVD (Matrix Factorization with BPR)

Pure dot-product MF with Xavier-initialised embeddings, trained with BPR loss on implicit positives.  
**Hyperparameters:** factors=64, lr=0.001, weight_decay=1e-5, epochs=50

In [ ]:
# ── Model ─────────────────────────────────────────────────────────────────────
class MatrixFactorization(nn.Module):
    def __init__(self, n_users, n_items, n_factors=64):
        super().__init__()
        self.user_emb = nn.Embedding(n_users, n_factors)
        self.item_emb = nn.Embedding(n_items, n_factors)
        nn.init.xavier_uniform_(self.user_emb.weight)
        nn.init.xavier_uniform_(self.item_emb.weight)

    def forward(self, users, items):
        return (self.user_emb(users) * self.item_emb(items)).sum(dim=1)

# ── Train ─────────────────────────────────────────────────────────────────────
svd_model = MatrixFactorization(n_users, n_items, n_factors=64).to(device)
optimizer = torch.optim.Adam(svd_model.parameters(), lr=0.001, weight_decay=1e-5)

EPOCHS = 50
for epoch in range(EPOCHS):
    svd_model.train()
    total_loss = 0
    for users, items in train_loader:
        neg_items    = sample_negatives_vectorized(users.cpu()).to(device)
        users, items = users.to(device), items.to(device)
        pos_scores   = svd_model(users, items)
        neg_scores   = svd_model(users, neg_items)
        loss         = bpr_loss(pos_scores, neg_scores)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss  += loss.item()
    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {total_loss/len(train_loader):.4f}")

# ── Evaluate ──────────────────────────────────────────────────────────────────
svd_results          = evaluate(svd_model, test_df, K=10, model_type="mf")
svd_results["model"] = "SVD"
print(f"\nSVD Results: {svd_results}")
# Expected: Recall@10=0.1457 | Precision@10=0.1158 | NDCG@10=0.1646 | HR@10=0.6168

## 4. Model 2 — NCF (Neural Collaborative Filtering with BPR)

Combines GMF (element-wise product) and MLP branches, concatenated before a final linear layer.  
**Hyperparameters:** factors=64, MLP layers=[128, 64, 32], dropout=0.2, lr=0.001, weight_decay=1e-5, epochs=50

In [ ]:
# ── Model ─────────────────────────────────────────────────────────────────────
class NCF(nn.Module):
    def __init__(self, n_users, n_items, n_factors=64, layers=[128, 64, 32]):
        super().__init__()
        # GMF embeddings
        self.gmf_user = nn.Embedding(n_users, n_factors)
        self.gmf_item = nn.Embedding(n_items, n_factors)
        # MLP embeddings
        self.mlp_user = nn.Embedding(n_users, n_factors)
        self.mlp_item = nn.Embedding(n_items, n_factors)
        nn.init.xavier_uniform_(self.gmf_user.weight)
        nn.init.xavier_uniform_(self.gmf_item.weight)
        nn.init.xavier_uniform_(self.mlp_user.weight)
        nn.init.xavier_uniform_(self.mlp_item.weight)
        # MLP tower
        mlp_input_size = n_factors * 2
        mlp_layers = []
        for out_size in layers:
            mlp_layers += [nn.Linear(mlp_input_size, out_size), nn.ReLU(), nn.Dropout(0.2)]
            mlp_input_size = out_size
        self.mlp    = nn.Sequential(*mlp_layers)
        self.output = nn.Linear(n_factors + layers[-1], 1)

    def forward(self, users, items):
        gmf     = self.gmf_user(users) * self.gmf_item(items)
        mlp_in  = torch.cat([self.mlp_user(users), self.mlp_item(items)], dim=1)
        mlp_out = self.mlp(mlp_in)
        return self.output(torch.cat([gmf, mlp_out], dim=1)).squeeze()

# ── Train ─────────────────────────────────────────────────────────────────────
ncf_model = NCF(n_users, n_items, n_factors=64, layers=[128, 64, 32]).to(device)
optimizer = torch.optim.Adam(ncf_model.parameters(), lr=0.001, weight_decay=1e-5)

EPOCHS = 50
for epoch in range(EPOCHS):
    ncf_model.train()
    total_loss = 0
    for users, items in train_loader:
        neg_items    = sample_negatives_vectorized(users.cpu()).to(device)
        users, items = users.to(device), items.to(device)
        pos_scores   = ncf_model(users, items)
        neg_scores   = ncf_model(users, neg_items)
        loss         = bpr_loss(pos_scores, neg_scores)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss  += loss.item()
    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {total_loss/len(train_loader):.4f}")

# ── Evaluate ──────────────────────────────────────────────────────────────────
ncf_results          = evaluate(ncf_model, test_df, K=10, model_type="ncf")
ncf_results["model"] = "NCF"
print(f"\nNCF Results: {ncf_results}")
# Expected: Recall@10=0.1298 | Precision@10=0.1068 | NDCG@10=0.1465 | HR@10=0.5810

## 5. Model 3 — LightGCN

Graph-based CF: propagates embeddings over a normalised user-item bipartite graph (no feature transformation, no activation).  
**Hyperparameters:** factors=64, layers=3, lr=0.001, weight_decay=1e-4, epochs=200, batch_size=4096  
Dense adjacency matrix used for correct gradient flow during training.

In [ ]:
# ── Build Dense Normalised Adjacency Matrix ───────────────────────────────────
def build_adj_dense(train_df, n_users, n_items, device):
    users = np.array([user2idx[u] for u in train_df["user_id"]])
    items = np.array([item2idx[i] for i in train_df["item_id"]])
    R     = sp.csr_matrix((np.ones(len(users)), (users, items)), shape=(n_users, n_items))
    upper = sp.hstack([sp.csr_matrix((n_users, n_users)), R])
    lower = sp.hstack([R.T, sp.csr_matrix((n_items, n_items))])
    A     = sp.vstack([upper, lower]).tocsr()
    deg   = np.array(A.sum(1)).flatten()
    d_inv = np.zeros_like(deg)
    d_inv[deg > 0] = deg[deg > 0] ** -0.5
    D     = sp.diags(d_inv)
    A_hat = (D @ A @ D)
    return torch.tensor(A_hat.toarray(), dtype=torch.float32).to(device)

# ── Model ─────────────────────────────────────────────────────────────────────
class LightGCN(nn.Module):
    def __init__(self, n_users, n_items, n_factors=64, n_layers=3):
        super().__init__()
        self.n_users  = n_users
        self.n_layers = n_layers
        self.E0       = nn.Embedding(n_users + n_items, n_factors)
        nn.init.normal_(self.E0.weight, std=0.1)

    def get_embeddings(self, adj):
        x   = self.E0.weight
        out = [x]
        for _ in range(self.n_layers):
            x = adj @ x          # dense matmul — gradients flow correctly
            out.append(x)
        out = torch.stack(out, dim=1).mean(dim=1)
        return out[:self.n_users], out[self.n_users:]

# ── Build adjacency + prepare full train tensors ───────────────────────────────
adj_dense       = build_adj_dense(train_df, n_users, n_items, device)
all_train_users = torch.tensor([user2idx[u] for u in train_df["user_id"]], dtype=torch.long)
all_train_items = torch.tensor([item2idx[i] for i in train_df["item_id"]], dtype=torch.long)

# ── Train ─────────────────────────────────────────────────────────────────────
lgcn_model = LightGCN(n_users, n_items, n_factors=64, n_layers=3).to(device)
optimizer  = torch.optim.Adam(lgcn_model.parameters(), lr=0.001, weight_decay=1e-4)

BATCH_SIZE = 4096
EPOCHS     = 200

for epoch in range(EPOCHS):
    lgcn_model.train()
    user_embs, item_embs = lgcn_model.get_embeddings(adj_dense)
    epoch_loss = 0
    optimizer.zero_grad()

    perm = torch.randperm(len(all_train_users))
    for start in range(0, len(all_train_users), BATCH_SIZE):
        batch_users = all_train_users[perm[start:start+BATCH_SIZE]].to(device)
        batch_items = all_train_items[perm[start:start+BATCH_SIZE]].to(device)
        neg_items   = sample_negatives_vectorized(batch_users.cpu()).to(device)

        pos_scores = (user_embs[batch_users] * item_embs[batch_items]).sum(dim=1)
        neg_scores = (user_embs[batch_users] * item_embs[neg_items]).sum(dim=1)
        loss       = bpr_loss(pos_scores, neg_scores)
        loss.backward(retain_graph=True)
        epoch_loss += loss.item()

    optimizer.step()    # single step per epoch after accumulating all gradients
    if (epoch + 1) % 20 == 0:
        n_batches = max(1, len(all_train_users) // BATCH_SIZE)
        print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {epoch_loss / n_batches:.4f}")

# ── Evaluate ──────────────────────────────────────────────────────────────────
lgcn_results          = evaluate_lgcn(lgcn_model, adj_dense, test_df, K=10)
lgcn_results["model"] = "LightGCN"
print(f"\nLightGCN Results: {lgcn_results}")
# Expected: Recall@10=0.0825 | Precision@10=0.0678 | NDCG@10=0.0928 | HR@10=0.4322

## 6. Results Comparison

In [ ]:
results_table = pd.DataFrame([svd_results, ncf_results, lgcn_results])
results_table = results_table.set_index("model")
results_table.index.name = "Model"
results_table.columns = ["Rec.@10", "Prec.@10", "NDCG@10", "HR@10"]

print(results_table.to_string())
print()
print(results_table.to_markdown())

## 7. Save Models & Results

In [ ]:
import pickle, json

# SVD model
with open("svd_model.pkl", "wb") as f:
    pickle.dump(svd_model.cpu(), f)

# ID encoders
with open("encoders.pkl", "wb") as f:
    pickle.dump({"user2idx": user2idx, "item2idx": item2idx,
                 "idx2user": idx2user, "idx2item": idx2item}, f)

# All results
with open("results.json", "w") as f:
    json.dump([svd_results, ncf_results, lgcn_results], f, indent=2)

print("Saved: svd_model.pkl | encoders.pkl | results.json")